# 02 · Train on FF++ + Celeb-DF (Colab T4)

Runs the full pipeline: unzip to local disk, build the manifest, benchmark the
step time, train, evaluate, export.

**Before you run this**, push the local fixes to GitHub — this notebook clones
the repo, so it needs them:

```
git add -A && git commit -m "T4 config, manifest and resume fixes" && git push
```

Checkpoints are written **to Drive**, not to `/content`. Colab will disconnect
during a multi-hour run; when it does, just re-run the training cell and it
picks up from the last completed epoch.

## 1 · Setup

In [ ]:
import torch, subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"

In [ ]:
# Only what training needs. facenet-pytorch and mediapipe are for face
# DETECTION, which you do not need here - the crops are already extracted.
# Skipping them also avoids their torch version pins fighting Colab's torch.
!pip install -q timm scikit-learn pandas tqdm

import os
REPO = "/content/deepfake_system"
if not os.path.exists(REPO):
    !git clone -q https://github.com/sailessawesome-ui/deepfake_system.git {REPO}
else:
    !cd {REPO} && git pull -q

WORK = f"{REPO}/deepfake_system"      # the package lives one level down
print("working dir:", WORK)
!ls {WORK}

## 2 · Data onto local disk

Never train off mounted Drive — its per-file latency is ~50-100 ms and you are
about to read 660,000 small JPEGs, many times over. Copy the zips across and
unzip locally. This takes a few minutes and only needs doing once per session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, time

# Paths from notebook 01. Left is the folder name config.DATA.sources expects.
ARCHIVES = {
    "FF++":          "/content/drive/MyDrive/FF++.zip",
    "celebdf_faces": "/content/drive/MyDrive/celebdf_faces.zip",
}

DATA_ROOT = "/content/data"
os.makedirs(DATA_ROOT, exist_ok=True)

for folder, zip_path in ARCHIVES.items():
    dest = os.path.join(DATA_ROOT, folder)
    if os.path.exists(dest) and os.listdir(dest):
        print(f"{folder}: already extracted, skipping")
        continue
    assert os.path.exists(zip_path), f"missing: {zip_path}"

    t0 = time.time()
    local_zip = f"/content/{os.path.basename(zip_path)}"
    print(f"{folder}: copying from Drive...", flush=True)
    shutil.copy(zip_path, local_zip)
    print(f"{folder}: unzipping...", flush=True)
    os.makedirs(dest, exist_ok=True)
    !unzip -q {local_zip} -d {dest}
    os.remove(local_zip)                     # reclaim the space immediately
    print(f"{folder}: done in {time.time()-t0:.0f}s")

print()
!df -h /content | tail -1
for folder in ARCHIVES:
    n = sum(len(f) for _, _, f in os.walk(os.path.join(DATA_ROOT, folder)))
    print(f"{folder:16s} {n:>9,} files")

## 3 · Build the manifest

`--dry-run` first: it scans 4,000 files per source and prints what it found, so
you can catch a labelling problem in seconds instead of after a full scan.

Splits come from the `train`/`val`/`test` folders already inside both archives.
That matters for Celeb-DF, whose folder names are hashes — identity cannot be
recovered from them, so the identity-based splitter would silently degrade into
a plain video split. Pass `--ignore-path-splits` to force identity grouping.

In [ ]:
!cd {WORK} && python -m data.build_manifest --root /content/data --dry-run

In [ ]:
!cd {WORK} && python -m data.build_manifest --root /content/data --out /content/manifest.csv

### Sanity check

Three things worth confirming before you spend six hours: no video appears in
two splits, both classes are present in every split, and the class balance is
what the sampler expects.

In [ ]:
import pandas as pd
from collections import Counter

df = pd.read_csv("/content/manifest.csv")
print(f"{len(df):,} frames over {df.video_id.nunique():,} videos")
print()

per_split = df.groupby(["split", "label"]).video_id.nunique().unstack(fill_value=0)
per_split.columns = ["real", "fake"]
per_split["total"] = per_split.real + per_split.fake
per_split["fake %"] = (100 * per_split.fake / per_split.total).round(1)
print("VIDEOS PER SPLIT")
print(per_split)
print()

print("VIDEOS PER SOURCE x SPLIT")
print(df.groupby(["source", "split"]).video_id.nunique().unstack(fill_value=0))
print()

print("METHODS")
print(df.groupby("method").video_id.nunique().sort_values(ascending=False))
print()

# --- the leak check ---
leaks = df.groupby("video_id").split.nunique()
leaks = leaks[leaks > 1]
print("=" * 60)
if len(leaks):
    print(f"FAIL: {len(leaks)} videos appear in more than one split")
    print(leaks.head())
else:
    print("PASS: every video sits in exactly one split")

ffpp = df[df.source == "ffpp"]
straddle = ffpp.groupby("identity").split.nunique()
straddle = straddle[straddle > 1]
print(f"FF++ identities spanning splits: {len(straddle)}"
      + ("  <-- faces shared across splits, numbers will be optimistic"
         if len(straddle) else "  (clean)"))

for s in ["train", "val", "test"]:
    sub = per_split.loc[s] if s in per_split.index else None
    if sub is None or sub.real == 0 or sub.fake == 0:
        print(f"WARNING: split '{s}' is missing one of the classes")
print("=" * 60)

## 4 · Benchmark before committing

Times a dozen real training steps and extrapolates. Run this before the long
job: it tells you whether the batch fits in 16 GB and roughly how many hours the
whole thing will take, in about a minute.

In [ ]:
import sys, time, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

sys.path.insert(0, WORK)
from config import DATA, MODEL, TRAIN
from data.dataset import ClipDataset, load_manifest, make_sampler
from models.net import build_model, param_groups

print(f"img_size {DATA.img_size} | clip_len {DATA.clip_len} | "
      f"batch {TRAIN.batch_size} | srm {MODEL.use_srm}")
print(f"images per step: {TRAIN.batch_size} x {DATA.clip_len} x 2 views = "
      f"{TRAIN.batch_size * DATA.clip_len * 2}")
print()

train_videos = load_manifest("/content/manifest.csv", splits=("train",))
bench_ds = ClipDataset(train_videos, train=True, clips_per_video=2, paired=True)
bench_ld = DataLoader(bench_ds, batch_size=TRAIN.batch_size,
                      sampler=make_sampler(train_videos),
                      num_workers=TRAIN.num_workers, pin_memory=True,
                      drop_last=True)

model = build_model(MODEL).cuda()
opt = torch.optim.AdamW(param_groups(model, TRAIN.lr, TRAIN.backbone_lr_mult,
                                     TRAIN.weight_decay))
scaler = torch.amp.GradScaler("cuda")
torch.cuda.reset_peak_memory_stats()

WARMUP, TIMED = 3, 12
times = []
model.train()
for i, batch in enumerate(bench_ld):
    if i >= WARMUP + TIMED:
        break
    torch.cuda.synchronize()
    t0 = time.time()
    clean = batch["clean"].cuda(non_blocking=True)
    degraded = batch["degraded"].cuda(non_blocking=True)
    y = batch["label"].cuda(non_blocking=True)
    with torch.autocast("cuda"):
        lc, _ = model(clean)
        ldg, _ = model(degraded)
        loss = (F.binary_cross_entropy_with_logits(lc, y) +
                F.binary_cross_entropy_with_logits(ldg, y))
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    if i >= WARMUP:
        times.append(time.time() - t0)
    print(f"  step {i}  {time.time()-t0:5.2f}s", flush=True)

s_per_step = sum(times) / len(times)
steps = len(bench_ld)
peak = torch.cuda.max_memory_allocated() / 1024**3
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print()
print("=" * 60)
print(f"median step      : {sorted(times)[len(times)//2]:.2f}s")
print(f"mean step        : {s_per_step:.2f}s")
print(f"steps per epoch  : {steps:,}")
print(f"epoch            : {steps * s_per_step / 60:.0f} min")
print(f"{TRAIN.epochs} epochs        : {steps * s_per_step * TRAIN.epochs / 3600:.1f} hours")
print()
print(f"peak VRAM        : {peak:.1f} / {total_gb:.1f} GB")
if peak > total_gb * 0.9:
    print("  TIGHT - drop batch_size to 6, or img_size to 192")
else:
    print("  comfortable")
print("=" * 60)
print()
print("If the total is more than ~10 hours, lower TRAIN.epochs in config.py.")
print("Training resumes across sessions, so a long run is survivable either way.")

del model, opt, scaler, bench_ld, bench_ds
torch.cuda.empty_cache()

## 5 · Train

Checkpoints go to Drive so a disconnect costs at most one epoch. The cell
auto-resumes: if `last.pt` exists it passes `--resume`, otherwise it starts
fresh. **When Colab drops you, just re-run this one cell.**

In [ ]:
import os

RUN = "/content/drive/MyDrive/deepfake_runs/v1"
os.makedirs(RUN, exist_ok=True)

last = f"{RUN}/last.pt"
resume_flag = f"--resume {last}" if os.path.exists(last) else ""
print("resuming from last.pt" if resume_flag else "starting fresh")

In [ ]:
!cd {WORK} && python train.py \
    --manifest /content/manifest.csv \
    --out {RUN} \
    {resume_flag}

### Watch the val numbers

`val f1` should climb quickly for the first few epochs, then flatten. If it sits
near 0.5 after three epochs something is wrong with the labels — go back to the
manifest sanity check.

A val F1 above ~0.97 here is expected and **is not the headline number**. These
splits are in-distribution. The number that means something is the unseen-source
one, which is what your held-out WildDeepfake set is for.

## 6 · Evaluate

`test` is the in-distribution number. `unseen_method` covers generator families
pulled out of training by `config.DATA.holdout_methods`.

In [ ]:
!cd {WORK} && python evaluate.py --checkpoint {RUN}/best.pt --manifest /content/manifest.csv --split test

In [ ]:
# Only meaningful once holdout_methods actually appear in your manifest -
# with FF++ and Celeb-DF alone this may be empty. It fills in when DF40 lands.
!cd {WORK} && python evaluate.py --checkpoint {RUN}/best.pt --manifest /content/manifest.csv --split unseen_method

## 7 · Export

Copy `best.pt` and `calibration.json` into `runs/v1/` next to the app, restart
the server, and the header chip flips from `baseline · no checkpoint` to
`model · tf_efficientnetv2_s`. Nothing else in the app changes.

In [ ]:
import os, shutil

EXPORT = "/content/drive/MyDrive/deepfake_export"
os.makedirs(EXPORT, exist_ok=True)

for f in ["best.pt", "calibration.json", "history.json",
          "report_test.json", "report_unseen_method.json"]:
    src = f"{RUN}/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"{EXPORT}/{f}")
        print(f"exported {f}  ({os.path.getsize(src)/1024**2:.1f} MB)")
    else:
        print(f"skip {f} (not produced)")

print()
print(f"Download from Drive: {EXPORT}")
print("Then locally:  cp best.pt calibration.json deepfake_system/runs/v1/")

---

## If something goes wrong

**OOM during training.** Lower `TRAIN.batch_size` to 6 or 4 in `config.py`, or
`DATA.img_size` to 192. Raise `accum_steps` to keep the effective batch near 32.
`train.py` holds the clean and degraded graphs simultaneously, so a step costs
`2 × batch × clip_len` images of activations — the memory scales with clip_len
just as hard as with batch size.

**Colab disconnected.** Re-run the setup cells (2, 3), the unzip cell (skips if
already extracted), then the training cell. It resumes from the last completed
epoch, with the optimizer, LR schedule and EMA intact.

**Session limit reached before 12 epochs.** Fine. Re-run the training cell in a
new session as many times as it takes. The only cost is redoing the unzip.

**`val f1` stuck near 0.5.** The labels are wrong. Re-run the manifest sanity
check and look at the per-method video counts.

**Dataloader is the bottleneck** (GPU utilisation low, step time dominated by
waiting). Colab free gives 2 vCPUs, and `TRAIN.num_workers` is set to 2 to
match. On Colab Pro with more cores, raise it.

## What this run does not give you

Everything here is in-distribution: you train and test on FF++ and Celeb-DF, so
97-99% at video level is the expected result and it is not evidence of
generalisation. The numbers that carry weight in a report are the ones from data
the model never saw — the held-out WildDeepfake source, and unseen generator
families once DF40 is in the manifest.

Keep `wild` out of training. `config.DATA.holdout_sources` already excludes it,
and it is the only honest generalisation number you will have.